# Case Study 1 — Carding Forums

**Carding** is the use and trade of stolen credit card data. Carding forums are online communities where criminals buy, sell, and exchange stolen card dumps, CVVs, fullz (full identity packages), and related services.

## Dataset

9 forum dumps spanning 2009–2021:

| Forum | Date | Notes |
|-------|------|-------|
| Carder.su | 2009-02 | One of the earliest major carding communities |
| Carding.biz | 2009-11 | |
| Carders.cc | 2010-12 | |
| Cardersplanet.biz | 2010-05 | |
| Carder.pro | 2013-04 | Russian-speaking, cp1251 encoding |
| Cardingmafia.ws | 2016-02 | |
| Crdshop.su | 2016-11 | |
| Elitecarders.name | 2016-08 | |
| CardingMafia / Cardmafia.cc | 2021-03 | Most recent dump |

All dumps are **vBulletin** SQL exports. Same schema across all forums — one parser handles everything.

## Analysis Plan

1. **Data loading** — parse all 9 dumps, inspect quality
2. **User overview** — registration patterns, post counts, contact details
3. **Timezone inference** — activity hours → UTC offset → likely country
4. **Cross-forum correlation** — same user appearing in multiple forums
5. **Stylometric analysis** — writing style embeddings, author clustering

---
## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../")  # make src/ importable from module root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from tqdm.notebook import tqdm
from pathlib import Path

from src.parsers.vbulletin import load_forum
from src.utils import list_forums, merge_tables, DATA_DIR, RESULTS_DIR
from src.timezone import build_user_timezone_profile, OFFSET_TO_REGION
from src.embeddings import embed_users, cosine_similarity

RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="darkgrid")

CATEGORY = "Carding Forums"
forum_zips = list_forums(CATEGORY)
print(f"Found {len(forum_zips)} forum dumps:")
for p in forum_zips:
    print(f"  {p.name}")

---
## 1. Data Loading

We parse all 9 dumps and merge each table into a single DataFrame.
Each row has a `forum` column so we always know its origin.

This may take 1–2 minutes — we're streaming several hundred MB of SQL.

In [ ]:
all_forums = []

for path in tqdm(forum_zips, desc="Parsing forums"):
    dfs = load_forum(path, tables=["user", "post", "pmtext", "userfield"])
    all_forums.append(dfs)

users    = merge_tables(all_forums, "user")
posts    = merge_tables(all_forums, "post")
pms      = merge_tables(all_forums, "pmtext")
fields   = merge_tables(all_forums, "userfield")

print(f"Users:  {len(users):,}")
print(f"Posts:  {len(posts):,}")
print(f"PMs:    {len(pms):,}")

In [ ]:
# Quick sanity check — how many users and posts per forum?
summary = pd.DataFrame({
    "users":  users.groupby("forum").size(),
    "posts":  posts.groupby("forum").size(),
    "pms":    pms.groupby("forum").size() if len(pms) > 0 else 0,
}).fillna(0).astype(int)

summary["posts_per_user"] = (summary["posts"] / summary["users"]).round(1)
summary.sort_values("users", ascending=False)

---
## 2. User Overview

In [ ]:
# Registration timeline — when were these forums active?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

users["joindate"].dropna().dt.year.value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Registrations by year", color="steelblue"
)

users["posts"].clip(upper=500).plot(
    kind="hist", bins=50, ax=axes[1],
    title="Post count distribution (capped at 500)",
    color="steelblue", logy=True
)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "carding_user_overview.png", dpi=150)
plt.show()

In [ ]:
# How many users left contact details?
# These are gold for attribution — ICQ numbers were frequently reused across platforms.
contact_cols = ["email", "icq", "skype", "aim", "yahoo", "msn", "homepage"]
available = [c for c in contact_cols if c in users.columns]

contact_fill = (
    users[available]
    .replace("", pd.NA)
    .notna()
    .sum()
    .sort_values(ascending=False)
)

contact_fill.plot(kind="barh", figsize=(8, 4), title="Users with contact details filled")
plt.xlabel("count")
plt.tight_layout()
plt.show()
print(contact_fill.to_string())

In [ ]:
# Top users by post count across all forums
# High post counts = high-value targets for deeper analysis
top_posters = (
    users[["forum", "username", "posts", "email", "icq", "skype", "joindate", "ipaddress"]]
    .sort_values("posts", ascending=False)
    .head(30)
    .reset_index(drop=True)
)
top_posters

---
## 3. Timezone Inference

We look at **when** each user posted (UTC hour) and find the timezone offset that best aligns their activity to normal waking hours (08:00–23:00 local time).

This gives us a probabilistic estimate of each user's location — without needing their IP address.

In [ ]:
# Only use visible posts with a valid timestamp
visible_posts = posts[posts["visible"].isin(["1", 1])].copy()

tz_profile = build_user_timezone_profile(visible_posts)

print(f"Profiled {len(tz_profile):,} users")
print(f"  with inferred offset: {tz_profile['inferred_utc_offset'].notna().sum():,}")
tz_profile.head()

In [ ]:
# Distribution of inferred UTC offsets across all users
offset_counts = (
    tz_profile["inferred_utc_offset"]
    .dropna()
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 4))
offset_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("UTC offset")
ax.set_ylabel("users")
ax.set_title("Inferred timezone distribution — Carding Forums")

# Annotate with region name
for i, (offset, count) in enumerate(offset_counts.items()):
    region = OFFSET_TO_REGION.get(int(offset), "")
    if count > offset_counts.max() * 0.05:  # only annotate significant bars
        ax.text(i, count + 5, region, ha="center", fontsize=7, rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "carding_timezone_distribution.png", dpi=150)
plt.show()

In [ ]:
# Top regions
region_counts = tz_profile["region"].value_counts()
print("Top inferred regions:")
print(region_counts.head(15).to_string())

In [ ]:
# Compare inferred offset vs self-reported timezoneoffset from the user table
# A mismatch is a signal that the user may have used a VPN or given false info
tz_combined = tz_profile.merge(
    users[["userid", "forum", "username", "timezoneoffset"]].assign(
        userid=lambda df: pd.to_numeric(df["userid"], errors="coerce")
    ),
    on="userid", how="left"
)

tz_combined["reported_offset"] = pd.to_numeric(tz_combined["timezoneoffset"], errors="coerce")
tz_combined["offset_mismatch"] = (
    tz_combined["inferred_utc_offset"].notna() &
    tz_combined["reported_offset"].notna() &
    (abs(tz_combined["inferred_utc_offset"] - tz_combined["reported_offset"]) > 2)
)

mismatch_rate = tz_combined["offset_mismatch"].mean()
print(f"Users with timezone mismatch (>2h): {tz_combined['offset_mismatch'].sum():,} ({mismatch_rate:.1%})")

# These are the most interesting — possible VPN users
(
    tz_combined[tz_combined["offset_mismatch"]]
    [["forum", "username", "reported_offset", "inferred_utc_offset", "region", "post_count"]]
    .sort_values("post_count", ascending=False)
    .head(20)
)

---
## 4. Cross-Forum Correlation

The same person often registers on multiple forums — buying on one, selling on another, seeking reputation across all of them.

We link users across forums using three signals, from most to least reliable:

| Signal | Reliability | Notes |
|--------|-------------|-------|
| Email address | High | Throwaway emails still match if the same one is reused |
| ICQ number | Very high | ICQ numbers are hard identifiers, rarely changed |
| Username | Medium | Common names collide; unique names are very reliable |
| Password hash | High | Same hash = same password = almost certainly same person |

In [ ]:
def find_cross_forum_matches(users_df: pd.DataFrame, field: str, min_len: int = 3) -> pd.DataFrame:
    """
    Find users that share the same value in `field` across different forums.
    Returns a DataFrame of matched groups.
    """
    df = users_df.copy()
    df[field] = df[field].replace("", pd.NA).str.strip().str.lower()
    df = df.dropna(subset=[field])
    df = df[df[field].str.len() >= min_len]  # ignore trivial values

    # Find values that appear in more than one forum
    multi = (
        df.groupby(field)["forum"]
        .nunique()
        .pipe(lambda s: s[s > 1])
        .index
    )

    matched = df[df[field].isin(multi)].copy()
    matched = matched.sort_values([field, "forum"])
    return matched


email_matches    = find_cross_forum_matches(users, "email")
icq_matches      = find_cross_forum_matches(users, "icq")
username_matches = find_cross_forum_matches(users, "username", min_len=4)
password_matches = find_cross_forum_matches(users, "password", min_len=10)

print(f"Email matches:    {email_matches[email_matches.columns[1]].nunique() if len(email_matches) else 0:,} unique values across >1 forum")
print(f"ICQ matches:      {icq_matches['icq'].nunique() if len(icq_matches) else 0:,}")
print(f"Username matches: {username_matches['username'].nunique() if len(username_matches) else 0:,}")
print(f"Password matches: {password_matches['password'].nunique() if len(password_matches) else 0:,}")

In [ ]:
# Show the top cross-forum identities by number of forums they appear in
cols = ["forum", "username", "email", "icq", "ipaddress", "joindate", "posts"]
available_cols = [c for c in cols if c in users.columns]

# Combine all match signals into a single unified profile
all_matches = pd.concat([
    email_matches.assign(match_signal="email"),
    icq_matches.assign(match_signal="icq") if len(icq_matches) else pd.DataFrame(),
    password_matches.assign(match_signal="password") if len(password_matches) else pd.DataFrame(),
], ignore_index=True)

if len(all_matches):
    # Users appearing in the most forums
    top = (
        all_matches.groupby("username")["forum"]
        .nunique()
        .sort_values(ascending=False)
        .head(10)
    )
    print("Top identities across multiple forums:")
    print(top.to_string())

In [ ]:
# Heatmap: how much user overlap is there between each pair of forums?
forum_names = users["forum"].unique()
overlap_matrix = pd.DataFrame(0, index=forum_names, columns=forum_names)

# Use lowercase username as a simple shared-identity signal
users["username_lower"] = users["username"].str.lower().str.strip()

forum_users = {f: set(g["username_lower"].dropna()) for f, g in users.groupby("forum")}

for f1 in forum_names:
    for f2 in forum_names:
        if f1 != f2:
            shared = len(forum_users[f1] & forum_users[f2])
            overlap_matrix.loc[f1, f2] = shared

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(overlap_matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_title("Shared usernames between forums (case-insensitive)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "carding_forum_overlap.png", dpi=150)
plt.show()

---
## 5. Stylometric Analysis

Writing style is surprisingly hard to change consistently. Vocabulary choices, sentence length, punctuation habits, and even typos tend to be stable across posts — even when a user changes their username.

We embed each user's combined posts into a single vector using `qwen3-embedding`. Users with similar writing styles will be close together in this embedding space.

**This requires Ollama running with qwen3-embedding pulled.** If not ready, skip to the end.

> `ollama pull qwen3-embedding` takes ~300MB and a few minutes.

In [ ]:
# Only embed users with enough text to produce a meaningful style signal.
# We use min_posts=10 here — more data = more reliable embedding.
# Feel free to lower to 5 if you want broader coverage.
MIN_POSTS = 10

user_ids, embeddings = embed_users(posts, min_posts=MIN_POSTS)
print(f"Embedded {len(user_ids):,} users  (dim={embeddings.shape[1]})")

In [ ]:
# Reduce to 2D with UMAP for visualization
# UMAP preserves local structure — nearby points in 2D were nearby in 4096D.
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer.fit_transform(embeddings)

embed_df = pd.DataFrame({
    "userid": user_ids,
    "x": coords[:, 0],
    "y": coords[:, 1],
})

# Join user metadata for hover labels
embed_df["userid_num"] = pd.to_numeric(embed_df["userid"], errors="coerce")
embed_df = embed_df.merge(
    users[["userid", "forum", "username", "posts"]].assign(
        userid=lambda df: pd.to_numeric(df["userid"], errors="coerce")
    ),
    left_on="userid_num", right_on="userid", how="left"
)

embed_df.head()

In [ ]:
# Interactive UMAP scatter — hover to see username and forum
fig = px.scatter(
    embed_df,
    x="x", y="y",
    color="forum",
    hover_data=["username", "posts"],
    title="User writing style clusters — Carding Forums (UMAP)",
    opacity=0.6,
    width=900, height=650,
)
fig.update_traces(marker=dict(size=4))
fig.write_html(RESULTS_DIR / "carding_style_umap.html")
fig.show()

In [ ]:
# Find the most stylistically similar user PAIRS across different forums
# High cosine similarity + different forum + different username = strong attribution signal
sim_matrix = cosine_similarity(embeddings, embeddings)
np.fill_diagonal(sim_matrix, 0)  # ignore self-similarity

# Get forum label for each embedded user
uid_to_forum = users.set_index(
    pd.to_numeric(users["userid"], errors="coerce")
)["forum"].to_dict()
uid_to_name = users.set_index(
    pd.to_numeric(users["userid"], errors="coerce")
)["username"].to_dict()

records = []
for i in range(len(user_ids)):
    for j in range(i + 1, len(user_ids)):
        uid_i = pd.to_numeric(user_ids[i], errors="coerce")
        uid_j = pd.to_numeric(user_ids[j], errors="coerce")
        f_i = uid_to_forum.get(uid_i, "")
        f_j = uid_to_forum.get(uid_j, "")
        if f_i != f_j:  # only cross-forum pairs
            records.append({
                "user_a": uid_to_name.get(uid_i, uid_i),
                "forum_a": f_i,
                "user_b": uid_to_name.get(uid_j, uid_j),
                "forum_b": f_j,
                "cosine_similarity": float(sim_matrix[i, j]),
            })

style_matches = pd.DataFrame(records).sort_values("cosine_similarity", ascending=False)
print("Top stylistically similar cross-forum user pairs:")
style_matches.head(20)

---
## 6. Summary

Run this cell last to generate a compact findings report.

In [ ]:
print("=" * 60)
print("CARDING FORUMS — ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nForums analyzed:     {len(forum_zips)}")
print(f"Total users:         {len(users):,}")
print(f"Total posts:         {len(posts):,}")
print(f"Total PMs:           {len(pms):,}")

top_region = tz_profile["region"].value_counts().idxmax() if len(tz_profile) else "N/A"
print(f"\nDominant region:     {top_region}")

if len(icq_matches):
    print(f"ICQ cross-matches:   {icq_matches['icq'].nunique():,}")
if len(email_matches):
    print(f"Email cross-matches: {email_matches['email'].nunique():,}")

print(f"\nResults saved to:    {RESULTS_DIR}")
print("=" * 60)